# SLAC XSIF to Bmad conversion

In [1]:
# Useful for debugging
%load_ext autoreload
%autoreload 2

In [2]:
# Patch in the slac2bmad package
import sys
sys.path.append('python')

In [3]:
from slac2bmad.xsif import prepare_xsif, remove_comment_blocks, replace_set, replace_set_commands, fix_matrix, expand_names, fix_names, unfold_comments, fold_comments
from slac2bmad.desplit import desplit_eles, desplit_ele
from slac2bmad.replace import replace_element, replace_eles
from slac2bmad.bmad import finalize_bmad

from glob import glob
import shutil

import subprocess
import json
import os

# Switches

In [4]:
INCLUDE_DEFERRED = True # Set to False before running `deferred.ipynb`

# Remove Comment blocks

# Change Set commands

In [5]:
replace_set('SET,  afa, afa, 1a')

#replace_set2('QUM1,   K1=')

'afa = afa, 1a'

# Expand names, correct matrix element syntax

In [6]:
fix_matrix('RM(3,4)')    

'R34'

In [7]:
expand_names('  afa APER BLMO')

'  afa aperture MONITOR'

In [8]:
fix_names(['sfafasfa safa APER RM(1,2)'])

['sfafasfa safa aperture R12']

# Folding and unfolding comments

In [9]:
def test():
    L0 = ['123\n', '123    !comment\n', '   \n', '  !simple comment\n', '123!456#789\n']
    L1 = unfold_comments(L0)
    L2 = fold_comments(L1)
    print(L0)
    print(L1)
    print(L2)
test()

['123\n', '123    !comment\n', '   \n', '  !simple comment\n', '123!456#789\n']
['123', '! INLINE--#    !comment', '123', '! EMPTY --#', '! SIMPLE --#  !simple comment', '! INLINE--#!456#789', '123']
['123', '123    !comment', '', '  !simple comment', '123!456#789']


# Desplitting (in Bmad)

In [10]:
line0 = 'qsx16_full: line = (qsx16, xcsx16, ycsx16, qsx16)'    
line1 = 'qsx16_full: line = (qsx16,  qsx16a)'  
line2 = 'qsx16_full: line = (qsx16)'  
print(desplit_ele(line2))

Info: line only has one ele: qsx16_full: line = (qsx16)
qsx16_full: line = (qsx16)


In [11]:
desplit_ele('WIG2H_full : LINE=(WIG2H1,YCWIGH,WIG2H2)')

Special desplit, names end with 1,2: WIG2H_full : LINE=(WIG2H1,YCWIGH,WIG2H2)
Desplitting ele: wig2h


'\n\n!Old split line:WIG2H_full : LINE=(WIG2H1,YCWIGH,WIG2H2)\nwig2h_full: line = (wig2h)\nycwigh[superimpose] = T\nycwigh[ref] = wig2h\n\n'

In [12]:
desplit_eles(['fafaa', line0])

Desplitting ele: qsx16


['fafaa',
 '\n\n!Old split line:qsx16_full: line = (qsx16, xcsx16, ycsx16, qsx16)\nqsx16_full: line = (qsx16)\nqsx16[L] = 2*qsx16[L]\nxcsx16[superimpose] = T\nxcsx16[ref] = qsx16\nycsx16[superimpose] = T\nycsx16[ref] = qsx16\n\n']

# Custom element replacements (in Bmad)

In [13]:
NEWELES = {}

NEWELES['umasxh'] = """
!------- SXR Undulator -------
my_umasxh_k = 5.0
umasxh: wiggler, 
        type = "VGHPU",
        L_period = 0.039, 
        n_period = 87, 
        b_max = my_umasxh_k * 2*pi*m_electron / (c_light * 0.039), 
        L = 87*0.039, 
        ds_step = 0.039*10
        
umasxh[L] = umasxh[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------
"""

NEWELES['umahxh'] = """
!------- HXR Undulator -------
my_umahxh_k = 2.0
umahxh: wiggler, 
        type = "HGVPU",
        L_period = 0.026, 
        n_period = 129, 
        b_max = my_umahxh_k * 2*pi*m_electron / (c_light * 0.026), 
        L = 129*0.026, 
        tilt=pi/2,
        ds_step = 0.026*10
        
umahxh[L] = umahxh[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------
    """




NEWELES['pssxh'] = """
!------- SXR Phase Shifter -------
!
! B_max = 2pi/lambda * sqrt(2*PHASE_INTEGRAL / L)
! 
pssxh_phase_integral = 3814e-9  !T^2 m^3, maximum, from: T^2mm^3 (180-3814)
pssxh_L        = 0.0825   ! m 
pssxh_L_period = 0.075 ! m 
pssxh: wiggler, type = "phase shifter", 
    L = pssxh_L,
    b_max = 2*pi / pssxh_L_period * sqrt(2 * pssxh_phase_integral / pssxh_L  ),
    n_period = 1
pssxh[L] = pssxh[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------
"""



NEWELES['pshxh'] = """
!------- HXR Phase Shifter -------
!
! B_max = 2pi/lambda * sqrt(2*PHASE_INTEGRAL / L)
! 
pshxh_phase_integral = 490e-9  !T^2 m^3, maximum, from: T^2mm^3 (80-490)
pshxh_L        = 0.0495 ! m 
pshxh_L_period = 0.045 ! m 
pshxh: wiggler, type = "phase shifter", 
    L = pshxh_L,
    b_max = 2*pi / pshxh_L_period * sqrt(2 * pshxh_phase_integral / pshxh_L  ),
    n_period = 1
pshxh[L] = pshxh[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------
"""



#-----------------
# XLEAP-II wigglers
NEWELES['umxl1h'] = """
!------- XLEAP-II wigglers -------
umxl0h: wiggler, 
        type = "LCLS-I",
        L_period = 0.55, 
        n_period = 6, 
        b_max = 0, ! = K * 2*pi*m_electron / (c_light * 0.55), 
        L = 6*0.55
        !ds_step = 0.55*10
                
umxl0h[L] = umxl0h[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------

umxl1h: umxl0h

"""

# Inherit from umxl0h
NEWELES['umxl2h'] = """
umxl2h: umxl0h
"""

NEWELES['umxl3h'] = """
umxl3h: umxl0h
"""

NEWELES['umxl4h'] = """
umxl4h: umxl0h
"""

# This needs to be extended
NEWELES['duqxl'] = """
! Extend to account for real WIGGLER elements for XLEAP
duqxl: drift, L = 0.2166 + 0.03
"""



In [14]:
# CU only replacements


CU_NEWELES = {}

CU_NEWELES['lh_und'] = """
!------- Laser Heater Undulator for Copper Linac -------
my_lh_und_k = 1.38523
lh_und: wiggler, 
        type = "laser_heater_undulator",
        L_period = 0.054, 
        n_period = 10, 
        b_max = my_lh_und_k * 2*pi*m_electron / (c_light * 0.054), 
         L = 10*0.054 ! Was: 0.506263, 
        ds_step = 0.054
        
lh_und[L] = lh_und[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------
    """

CU_NEWELES['dh03a'] = """
! Shorten so that lh_und has an integer number of poles
dh03a: drift, l = 0.09290825 - ( 10*0.054 - 0.506263 ) /2, type = "CSR"
"""

CU_NEWELES['dh03b'] = """
! Shorten so that lh_und has an integer number of poles
dh03b: drift, l = 0.08401830- ( 10*0.054 - 0.506263 ) /2, type = "CSR"
"""

# Add these replacements
CU_LINAC_REPLACEMENTS = json.load(open('replacements/good_cu_linac_replacements.json'))
for name, replace in CU_LINAC_REPLACEMENTS.items():
    CU_NEWELES[name.lower()+'_full'] = replace

# Add deferred elements    
if INCLUDE_DEFERRED:
    CU_NEWELES.update(json.load(open('replacements/deferred_cu_replacements.json')))
    
print(CU_NEWELES.keys())

dict_keys(['lh_und', 'dh03a', 'dh03b', 'l0a_full', 'l0b_full', 'k21_1b_full', 'k21_1c_full', 'k21_1d_full', 'l1x_full', 'k21_3b_full', 'k21_4a_full', 'k21_5a_full', 'k21_6a_full', 'k21_7a_full', 'k21_8a_full', 'k21_8d_full', 'k22_2a_full', 'k22_3a_full', 'k22_4a_full', 'k22_5a_full', 'k22_6a_full', 'k22_7a_full', 'k22_8a_full', 'k22_8d_full', 'k23_2a_full', 'k23_3a_full', 'k23_4a_full', 'k23_5a_full', 'k23_6a_full', 'k23_7a_full', 'k23_8a_full', 'k23_8d_full', 'k24_2a_full', 'k24_3a_full', 'k24_4a_full', 'k24_5a_full', 'k24_6a_full', 'k24_6d_full', 'k25_1a_full', 'k25_2a_full', 'k25_3a_full', 'k25_4a_full', 'k25_5a_full', 'k25_6a_full', 'k25_7a_full', 'k25_8a_full', 'k25_8d_full', 'k26_2a_full', 'k26_3a_full', 'k26_4a_full', 'k26_5a_full', 'k26_6a_full', 'k26_7a_full', 'k26_8a_full', 'k26_8d_full', 'k27_2a_full', 'k27_3a_full', 'k27_4a_full', 'k27_5a_full', 'k27_6a_full', 'k27_7a_full', 'k27_8a_full', 'k27_8d_full', 'k28_2a_full', 'k28_3a_full', 'k28_4a_full', 'k28_5a_full', 'k28_6a_fu

In [15]:
# SC Only replacements

SC_NEWELES = {}

SC_NEWELES['umhtr'] = """
!------- Laser Heater Undulator for SC Linac -------
my_umhtr_k = 0.960143

umhtr: wiggler, 
        type = "laser_heater_undulator",
        L_period = 0.054, 
        n_period = 10, 
        b_max = my_umhtr_k * 2*pi*m_electron / (c_light * 0.054), 
        L = 10*0.054 ! Was: 0.506263, 
        ds_step = 0.054
        
umhtr[L] = umhtr[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------
    """
# 
SC_NEWELES['dh02c'] = """
! Shorten so that umhtr has an integer number of poles
dh02c: drift, l = 0.2795065 - ( 10*0.054 - 0.506263 ) /2 , type = "CSR" !0.297036
"""

SC_NEWELES['dh02d'] = """
! Shorten so that umhtr has an integer number of poles
dh02d: drift, l = 0.2724707 - ( 10*0.054 - 0.506263 ) /2, type = "CSR" !0.2900002
"""

# Not needed. Desplitting handles cavities now.
# Add these repalcements
#SC_LINAC_REPLACEMENTS = json.load(open('replacements/good_sc_linac_replacements.json'))
#for name, replace in SC_LINAC_REPLACEMENTS.items():
#    SC_NEWELES[name.lower()+'_full'] = replace


# Add deferred elements 
if INCLUDE_DEFERRED:
    SC_NEWELES.update(json.load(open('replacements/deferred_sc_replacements.json')))
    
print(SC_NEWELES.keys())


dict_keys(['umhtr', 'dh02c', 'dh02d', 'RFB0H00', 'RFB0H04', 'RFB0H08', 'RFBHD00', 'RFBHD04', 'RFBDG001', 'YCDGTCY', 'XCDGTCY', 'YCDGTCX', 'XCDGTCX', 'OTRDG01', 'OTRDG01_YAG', 'RFBDG002', 'OTRDG03', 'OTRDG03_YAG', 'RFBDG003', 'TCYDG0', 'TCXDG0', 'WSC006', 'RFBC006', 'CYC02', 'CXC02', 'RFBC011', 'RFB1C01', 'WS11B', 'SQ13B', 'BZC1', 'RFBC104', 'RFBC106', 'CYC12', 'CXC12', 'RFBC108', 'RFBC110', 'RFB2C00', 'RFB2C01', 'WS21B', 'RFBX02', 'SDOG1', 'WSDOG', 'SDOG2', 'OTR31', 'OTR31_YAG', 'RFBWSBP1', 'RFBWSBP2', 'RFBWSBP3', 'RFBWSBP4', 'RFBBP24', 'RFBBP35', 'BKYSP0S', 'BKYSP4S', 'BKYSP5S', 'SSP1S', 'SSP2S', 'IMBCSS1', 'IMBCSS2', 'RFBBP30', 'RFBBP33', 'SDL1', 'SDL2', 'PCTDKIK3S', 'PCTDKIK4S', 'RFBEM4B', 'RFBE32B', 'RFBE34B', 'RFBE36B', 'TRUE1B', 'PCTCXB', 'TCX01B', 'TCX02B', 'SPTCXB', 'RFBDDB', 'WSDUMPB', 'BKYSP0H', 'BKYSP4H', 'BKYSP5H', 'SSP1H', 'SSP2H', 'IMBCSH1', 'IMBCSH2', 'RFBBSYQ3', 'PCBSYH', 'RFBBSYQ6', 'BCX311', 'BCX312', 'BCX313', 'BCX314', 'RFBDL1', 'WIG1H', 'YCWIGH', 'WIG3H', 'RFBEM4',

In [16]:
print("!---------------------\n! CAVL015 LCAVITY\nCAVL015_full: line = (CAVL015)\n! contains zero length elements:\n    CSP01[superimpose] = T\n    CSP01[ref] = CAVL015\n    CSP01[ref_origin] = beginning\n    CSP01[offset] = 0.659221562\n\n")

!---------------------
! CAVL015 LCAVITY
CAVL015_full: line = (CAVL015)
! contains zero length elements:
    CSP01[superimpose] = T
    CSP01[ref] = CAVL015
    CSP01[ref_origin] = beginning
    CSP01[offset] = 0.659221562




In [17]:
def all_replacements(master_file):
    dat = {}
    dat.update(NEWELES)
    if master_file.startswith('CU_'):
        print('CU replacements')
        dat.update(CU_NEWELES)
        return dat
    elif master_file.startswith('SC_'):
        print('SC replacements')
        dat.update(SC_NEWELES)
        return dat
    else:
        raise 
#all_replacements('CU_')        

# Full conversion

In [18]:
#prepare_xsif('LTU.xsif')

In [19]:
#translate_xsif_to_bmad('LTU.xsif')

In [20]:
#with open('LTU.bmad') as f:
#    lines = f.readlines()

In [21]:
#lines2 = replace_eles(lines, NEWELES)

In [22]:
#NEWELES.keys()

In [23]:

#finalize_bmad('UND.bmad', replacements=NEWELES)            

# Convert all

In [24]:
!mkdir temp

In [25]:
# Clean
!rm *xsif *bmad *digested*

zsh:1: no matches found: *xsif


In [26]:
!pwd

/Users/chrisonian/Code/GitHub/lcls-lattice/bmad/conversion


In [27]:
!cp $LCLS_LATTICE/mad/*xsif .

In [28]:
XSIF_FILES=[f for f in os.listdir() if f.endswith('.xsif')]
for f in XSIF_FILES:
    prepare_xsif(f, save=False)

Preparing LCLS_L3.xsif
Preparing INJ.xsif
Preparing CU_SXR.xsif
Preparing LCLS2cu_master.xsif
Preparing LCLS_L2.xsif
Preparing CU_SFTH.xsif
Preparing ALINE.xsif
Preparing DASEL.xsif
Preparing CUSXR.xsif
Preparing DIAG0.xsif
Preparing SPRD.xsif
Preparing HXTES.xsif
Preparing UNDtemp.xsif
Preparing DLBM.xsif
Preparing SC_SXR.xsif
Preparing CM.xsif
Preparing BC1.xsif
Preparing common.xsif
Preparing BC2.xsif
Preparing LCLS2cu.xsif
Preparing SFT.xsif
Preparing LCLS_L3e.xsif
Preparing LCLS2sc_master.xsif
Preparing BSYsc.xsif
Preparing SC_DASEL.xsif
Preparing UND.xsif
Preparing CU_SPEC.xsif
Preparing SC_DIAG0.xsif
Preparing CU_GSPEC.xsif
Preparing SXTES.xsif
Preparing CU_HXR.xsif
Preparing LCLS_L1e.xsif
Preparing LCLS_L1.xsif
Preparing BYP.xsif
Preparing SC_HXR.xsif
Preparing EXT.xsif
Preparing BSYcu.xsif
Preparing SC_SFTS.xsif
Preparing CU_ALINE.xsif
Preparing SC_BSYD.xsif
Preparing LCLS_L2e.xsif
Preparing LTU.xsif


In [29]:
!mv *xsif temp/

In [30]:
CU_MASTERS = [f for f in os.listdir('../../mad') if f.startswith('CU_')]
SC_MASTERS = [f for f in os.listdir('../../mad') if f.startswith('SC_')]
CU_MASTERS, SC_MASTERS

(['CU_SXR.xsif',
  'CU_SFTH.xsif',
  'CU_SPEC.xsif',
  'CU_GSPEC.xsif',
  'CU_HXR.xsif',
  'CU_ALINE.xsif'],
 ['SC_SXR.xsif',
  'SC_DASEL.xsif',
  'SC_DIAG0.xsif',
  'SC_HXR.xsif',
  'SC_SFTS.xsif',
  'SC_BSYD.xsif'])

In [31]:
TEMPDIR = './temp/'
WORKDIR = './work/'

In [32]:
!mkdir {TEMPDIR}
!mkdir {WORKDIR}

mkdir: ./temp/: File exists


# Process 

In [33]:
DEST = os.path.expandvars('$LCLS_LATTICE/bmad/master/')

In [34]:
def process_master(master):
    
    print(f'Converting {master}')
    
    shutil.copytree(TEMPDIR, WORKDIR, dirs_exist_ok=True)
    
    # New method
    SCRIPT = f'python $ACC_ROOT_DIR/util_programs/mad_to_bmad/mad8_to_bmad.py --no_prepend_vars -f {master}'

    res = subprocess.run(SCRIPT, shell=True, cwd=WORKDIR)
    
    assert res.returncode == 0
    
    BMAD_FILES=glob(WORKDIR+'/*bmad')

    REPLACEMENTS = all_replacements(master)

    for f in BMAD_FILES:
        finalize_bmad(f, replacements=REPLACEMENTS, verbose=False)   
    
    print(f'    Copying all to {DEST}')
    for f in BMAD_FILES:
        #print(f'copying {f} to {DEST}')
        shutil.copy(f, DEST)
    
process_master('SC_SXR.xsif')

Converting SC_SXR.xsif
Input lattice file is:  SC_SXR.xsif
Output lattice file is: SC_SXR.bmad
SC replacements
    Copying all to /Users/chrisonian/Code/GitHub/lcls-lattice/bmad/master/


In [35]:
for m in CU_MASTERS:
    process_master(m)

Converting CU_SXR.xsif
Input lattice file is:  CU_SXR.xsif
Output lattice file is: CU_SXR.bmad
CU replacements
    Copying all to /Users/chrisonian/Code/GitHub/lcls-lattice/bmad/master/
Converting CU_SFTH.xsif
Input lattice file is:  CU_SFTH.xsif
Output lattice file is: CU_SFTH.bmad
CU replacements
    Copying all to /Users/chrisonian/Code/GitHub/lcls-lattice/bmad/master/
Converting CU_SPEC.xsif
Input lattice file is:  CU_SPEC.xsif
Output lattice file is: CU_SPEC.bmad
CU replacements
    Copying all to /Users/chrisonian/Code/GitHub/lcls-lattice/bmad/master/
Converting CU_GSPEC.xsif
Input lattice file is:  CU_GSPEC.xsif
Output lattice file is: CU_GSPEC.bmad
CU replacements
    Copying all to /Users/chrisonian/Code/GitHub/lcls-lattice/bmad/master/
Converting CU_HXR.xsif
Input lattice file is:  CU_HXR.xsif
Output lattice file is: CU_HXR.bmad
CU replacements
    Copying all to /Users/chrisonian/Code/GitHub/lcls-lattice/bmad/master/
Converting CU_ALINE.xsif
Input lattice file is:  CU_ALINE.

In [36]:
for m in SC_MASTERS:
    process_master(m)

Converting SC_SXR.xsif
Input lattice file is:  SC_SXR.xsif
Output lattice file is: SC_SXR.bmad
SC replacements
    Copying all to /Users/chrisonian/Code/GitHub/lcls-lattice/bmad/master/
Converting SC_DASEL.xsif
Input lattice file is:  SC_DASEL.xsif
Output lattice file is: SC_DASEL.bmad
SC replacements
    Copying all to /Users/chrisonian/Code/GitHub/lcls-lattice/bmad/master/
Converting SC_DIAG0.xsif
Input lattice file is:  SC_DIAG0.xsif
Output lattice file is: SC_DIAG0.bmad
SC replacements
    Copying all to /Users/chrisonian/Code/GitHub/lcls-lattice/bmad/master/
Converting SC_HXR.xsif
Input lattice file is:  SC_HXR.xsif
Output lattice file is: SC_HXR.bmad
SC replacements
    Copying all to /Users/chrisonian/Code/GitHub/lcls-lattice/bmad/master/
Converting SC_SFTS.xsif
Input lattice file is:  SC_SFTS.xsif
Output lattice file is: SC_SFTS.bmad
SC replacements
    Copying all to /Users/chrisonian/Code/GitHub/lcls-lattice/bmad/master/
Converting SC_BSYD.xsif
Input lattice file is:  SC_BSYD

# Final cleanup

In [37]:
!rm -r {TEMPDIR}
!rm -r {WORKDIR}